In [ ]:
def check_and_pick(parent: Path, pattern: str) -> Path:
    matches = tuple(parent.glob(pattern))
    if not matches:
        raise ValueError(f"A file matching the pattern {str(parent / pattern)} is expected but missing.")
    f, *extra_matches = matches
    if extra_matches:
        raise ValueError(
            f"Found unexpected extra matches for glob pattern {str(parent / pattern)}."
            f"File with pattern {pattern} should be unique"
        )
    return f

def get_source_coords(sourceID: str, catalogue: CatalogueLoader) -> CoordEquatorial:
    """Extracts the source RA/Dec coords from catalogue."""
    data = source_catalogue_data(sourceID, catalogue.DLdata)
    return CoordEquatorial(data['RA'], data['DEC'])

def equatorial2angle(
    sdl: DataLoader,
    camera: CodedMaskCamera,
    ra: float,
    dec: float,
) -> tuple[float, float]:
    """Converts RA/Dec coords to camera local-frame angular coords."""
    shifts = equatorial2shift(sdl, camera, ra, dec)
    return tuple(shift2angle(camera, s) for s in shifts)

def find_boxmax(sky: NDArray, centre: tuple[int, int], boxsize: int = 5) -> tuple[int, int]:
    """Finds the argmax of given sky in a box centered around `pos`."""
    rcut, ccut = map(lambda idx: slice(idx - boxsize, idx + boxsize + 1), centre)
    p, q = centre
    n, m = argmax(sky[rcut, ccut])
    return p - boxsize + n, q - boxsize + m

In [ ]:
from bloodmoon.coords import pos2shift
from IROSrec.iros.optim import solver


def config_optimiser(
    camera: CodedMaskCamera,
    vignetting: bool | Callable[[CodedMaskCamera, NDArray, float, float], NDArray],
    psfy: bool | Callable[[CodedMaskCamera, NDArray], NDArray],
    sg_comp_weights: dict[str, dict[str, float]],
    fit_weights: NDArray | Callable[[NDArray], NDArray] | None = None,
    camera_coding_power: float = 0.85,
    psf_crop_offset: tuple[int, int] = (0, 0),
    verbose: bool = False,
) -> Callable[[NDArray, tuple[int, int]], OptResult]:
    """
    Configures source params optimiser 
    """
    def process_skyimg(
        sky: NDArray,
        pos: tuple[int, int],
    ) -> NDArray:
        """
        Processes the sky image for optimisation.
        """
        i, j = pos
        offset_y, offset_x = psf_crop_offset
        psf_slit_px_y, psf_slit_px_x = (
            int(camera.specs.slit_deltay * camera.upscale_f.y / camera.specs.mask_deltay),
            int(camera.specs.slit_deltax * camera.upscale_f.x / camera.specs.mask_deltax),
        )
        crop_y, crop_x = (psf_slit_px_y + offset_y, psf_slit_px_x + offset_x)
        slice_y, slice_x = (slice(i - crop_y, i + crop_y + 1), slice(j - crop_x, j + crop_x + 1))
        cropped = sky[slice_y, slice_x]
        return cropped.flatten()

    def _ModelShiftFluence(arg_sky: tuple[int, int]) -> Callable[[NDArray, float, float, float], NDArray]:
        """
        Initialises the source model.
        """
        def f(x: NDArray, shift_x: float, shift_y: float, fluence: float) -> NDArray:
            """Models the source sky image."""
            proj = np.sign(shift_x) * np.cos(np.deg2rad(shift2angle(camera, shift_x)))

            sg = sum(
                tuple(
                    eband['wcts'] * fluence * model_shadowgram(camera, shift_x + proj * eband['dshift'], shift_y, vignetting, psfy)
                    for eband in sg_comp_weights.values()
                )
            )
            
            modeled = decode(camera, sg)
            return process_skyimg(modeled, arg_sky)
        
        return f

    def optimiser(
        sky: NDArray,
        arg_sky: tuple[int, int],
    ) -> OptResult:
        """
        Performs the optimization to fit a point source model to sky image data.
        """
        model_func = _ModelShiftFluence(arg_sky)
        sky_peak = sky[*arg_sky]

        # setup solver params
        # * setup source data + std
        sky_ydata = process_skyimg(sky, arg_sky)
        if fit_weights is not None:
            weights = (
                fit_weights if isinstance(fit_weights, np.ndarray)
                else fit_weights(sky_ydata)
            )
        else:
            weights = np.ones_like(sky_ydata)
        # * extract source coords and counts starting values
        sx_start, sy_start = pos2shift(camera, *arg_sky)
        cts_start = (
            sky_peak / camera_coding_power if psfy is not False else sky_peak
        )
        start_params_vals = np.array([sx_start, sy_start, cts_start])
        # * setup fit params boundaries
        #    - the shifts are allowed to fluctuate in a small pixel box since
        #      the extracted position is close enough to the true source pos
        #      A small box also account for superimposed or close sources,
        #      which may introduce biases in the source fit procedure
        #    - the box is built from the digital upsampling since in the worst
        #      case (source at 45 deg wrt optical axis) the high energy detected
        #      photons median absorption distance is 225um. The projection of this
        #      distance on the camera plane is always smaller than `px_size / ups`  
        #    - the fluence cannot be smaller than the one observed at the peak,
        #      and we insert a lower value just for precaution (if simulating
        #      for example an infinite detector spatial resolution)
        px_dim_x, px_dim_y = (
            camera.specs.mask_deltax / camera.upscale_f.x,
            camera.specs.mask_deltay / camera.upscale_f.y,
        )
        bounds = [
            (
                max(sx_start - camera.upscale_f.x * px_dim_x, camera.bins_sky.x[0]),
                max(sy_start - camera.upscale_f.y * px_dim_y, camera.bins_sky.y[0]),
                0.95 * sky_peak,
            ),
            (
                min(sx_start + camera.upscale_f.x * px_dim_x, camera.bins_sky.x[-1]),
                min(sy_start + camera.upscale_f.y * px_dim_y, camera.bins_sky.y[-1]),
                1.4 * sky_peak,
            ),
        ]
        # perform optimisation
        slvr_kwargs = {
            # - solver kwargs
            'p0': start_params_vals,
            'sigma': weights,
            'bounds': bounds,
            'method': 'trf',
            # - least square kwargs
            'jac': '3-point',
            'ftol': 1e-8,
            'xtol': 1e-8,
            'x_scale': 'jac',
            'loss': 'linear',
        }
        result = solver(
            func=model_func,
            xdata=np.arange(len(sky_ydata)),
            ydata=sky_ydata,
            verbose=verbose,
            **slvr_kwargs,
        )

        return result
    
    return optimiser

def config_fitter(
    optimiser: Callable[[NDArray, tuple[int, int]], OptResult],
) -> Callable[[tuple[int, int], NDArray], tuple[NDArray, NDArray]]:
    """
    Configures source candidates parameters fitter.
    """
    def fitter(arg_sky: tuple[int, int], sky: NDArray) -> tuple[NDArray, NDArray]:
        """Performs the optimisation of the source candidate params."""
        try:
            params: OptResult = optimiser(
                sky=sky,
                arg_sky=arg_sky,
            )
        except Exception as e:
            raise RuntimeError(f"Optimization failed: {str(e)}") from e
        
        return params.params, params.covar
    
    return fitter

In [ ]:
srcID = 's17'

mask = (cat.DLdata['NAME'] != srcID)
exclude_srcs: list[CoordEquatorial] = [
    CoordEquatorial(ra, dec) for ra, dec in zip(cat.DLdata['RA'][mask], cat.DLdata['DEC'][mask])
]
phs = ds.filter_data(sdl.DLdata, E_min=None, E_max=None, coords=exclude_srcs)

detector, _ = count(wfm, phs, rot_angle=rot_angle)
skymap = decode(wfm, detector)
varmap = variance(wfm, detector)

In [ ]:
ra, dec = get_source_coords(srcID, cat)
src_pos = equatorial2pos(sdl, wfm, ra, dec)
src_peak = find_boxmax(skymap, src_pos, boxsize=2 * wfm.upscale_f.x + 1)

In [ ]:
rows, cols = crop_source_psf(wfm, src_peak, 2 + wfm.upscale_f.x, 5 * wfm.upscale_f.y)
psfvar = varmap[rows, cols]
fit_weights = np.pow(np.clip(psfvar, a_min=1.0, a_max=None), -0.5).flatten()

In [ ]:
sg_comp_weights = {
    (2, 6): {'wcts': 0.85, 'dshift': 5.7e-3},
    (6, 15): {'wcts': 0.14, 'dshift': 7.8e-2},
    (15, 50): {'wcts': 0.01, 'dshift': 2.71e-1},
}

optimiser = config_optimiser(
    camera=wfm,
    vignetting=vignetting,
    psfy=psfy,
    sg_comp_weights=sg_comp_weights,
    fit_weights=fit_weights,
    psf_crop_offset=(5 * wfm.upscale_f.y, 2 + wfm.upscale_f.x),
    verbose=True,
)
fit_src_pars = config_fitter(optimiser)

In [ ]:
pars, pcov = fit_src_pars(src_peak, skymap)

In [ ]:
pars, np.sqrt(np.diag(pcov))

In [ ]:
# NOTE: check also in Darksun_Reconstruction

#     * demo_source_obs_spectrum.ipynb
#     * demo_detector_absrp.ipynb
#     * demo_sources_ebands_fit_benchmrk.ipynb